In [1]:
import pandas as pd
import numpy as np

In [2]:
cluster1995 = pd.read_csv(r"C:\Users\ayaan\OneDrive - London School of Economics\Academics\MPA_DSPP\Moody's - Capstone Project\Capstone-Ayaan-ASUS\MASTER\clustersagg.csv")
master = pd.read_csv(r"C:\Users\ayaan\OneDrive - London School of Economics\Academics\MPA_DSPP\Moody's - Capstone Project\Capstone-Ayaan-ASUS\MASTER\Master.csv", index_col=0)

In [3]:
include_list = ['AGO', 'ARE', 'AZE', 'BFA', 'BHR', 'BOL', 'CHL', 'CIV', 'CMR',
       'COD', 'COG', 'DZA', 'ECU', 'EGY', 'ETH', 'GAB', 'GHA', 'GIN',
       'GNQ', 'IDN', 'IRN', 'IRQ', 'KAZ', 'KEN', 'KWT', 'LAO', 'LBR',
       'LBY', 'MDG', 'MLI', 'MMR', 'MNG', 'MOZ', 'MWI', 'MYS', 'NER',
       'NGA', 'OMN', 'PNG', 'QAT', 'RUS', 'RWA', 'SAU', 'TCD', 'TGO',
       'TTO', 'TZA', 'UGA', 'UZB', 'VEN', 'VNM', 'YEM', 'ZMB', 'ZWE']

In [4]:
master = master[(master["Country Code"].isin(include_list))]

In [5]:
master = pd.merge(master, cluster1995[["Country Code", "Cluster"]], on="Country Code", how= "left")
master.head()

,Country Code,Country Name,Year,Access to electricity (% of population),Adjusted savings: gross savings (% of GNI),Agriculture,Capital depreciation rate,Clientelism index,"Death rates, crude per 1000 people",Domestic credit to private sector (% of GDP),...,"Use of IMF credit (DOD, current US$)",Total_Production,Total_Reserves,Total_Production_Value,Total_Reserves_Value,Hydrocarbons_Dominant,Subsoil_Metals_Dominant,Precious_Metals_Dominant,Population,Cluster
0,AGO,Angola,1995,24.2,48.055414,9.386791,0.035367,0.84,18.700,22.274928,...,0.0,230991930.0,1140625.0,4.573759e+09,2.258496e+07,1,0,0,13699778.0,2
1,AGO,Angola,1996,24.2,48.055414,9.386791,0.038398,0.84,18.445,22.274928,...,0.0,261331263.9,1348675.0,5.894739e+09,3.042149e+07,1,0,0,14170973.0,2
2,AGO,Angola,1997,24.2,48.055414,9.386791,0.040405,0.84,18.184,22.274928,...,0.0,270465000.0,1423500.0,5.405294e+09,2.844892e+07,1,0,0,14660413.0,2
3,AGO,Angola,1998,24.2,48.055414,9.386791,0.040825,0.84,18.925,22.274928,...,0.0,266760000.0,1470950.0,3.392030e+09,1.870410e+07,1,0,0,15159370.0,2
4,AGO,Angola,1999,24.2,48.055414,9.386791,0.041406,0.84,18.518,22.274928,...,374707057.4,271947000.0,1843250.0,4.689445e+09,3.178494e+07,1,0,0,15667235.0,2


In [7]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ============ CONFIGURATION ============
COL_COUNTRY_CODE = 'Country Code'
COL_COUNTRY_NAME = 'Country Name'
COL_YEAR = 'Year'
COL_ECI = 'Economic Complexity Index'
COL_GDP_PC = 'GDP per capita (constant prices, PPP)'
COL_PRODUCTION = 'Total_Production_Value'
COL_CLUSTER = 'Cluster'

CLUSTER_COLORS = {
    0: '#2ecc71', 1: '#e74c3c', 2: '#f39c12',
    3: '#9b59b6', 4: '#3498db'
}
CLUSTER_NAMES = {
    0: 'Oil, Few Minerals',
    1: 'No Oil, No Minerals',
    2: 'Some Oil, No Minerals',
    3: 'Minerals, No Oil',
}


def create_rosling_chart_arrows(df, arrow_opacity=0.5, arrow_width=2):
    """
    Hans Rosling-style chart with arrows from 1995 origin to current year.
    """
    data = df.copy()
    
    # Preprocessing
    data['Log GDP per capita'] = np.log(data[COL_GDP_PC])
    data['Production_Per_Capita'] = data[COL_PRODUCTION] / data['Population']
    
    # Fix cluster colors to 1995 values
    cluster_1995 = data[data[COL_YEAR] == 1995][[COL_COUNTRY_CODE, COL_CLUSTER]].copy()
    cluster_1995 = cluster_1995.rename(columns={COL_CLUSTER: 'Cluster_1995'})
    data = data.merge(cluster_1995, on=COL_COUNTRY_CODE, how='left')
    data = data.dropna(subset=['Cluster_1995', 'Log GDP per capita', COL_ECI, 'Production_Per_Capita'])
    data['Cluster_1995'] = data['Cluster_1995'].astype(int)
    
    # Bubble sizing
    data['Bubble_Size'] = np.sqrt(data['Production_Per_Capita'])
    min_s, max_s = data['Bubble_Size'].min(), data['Bubble_Size'].max()
    data['Bubble_Size_Scaled'] = 8 + (data['Bubble_Size'] - min_s) / (max_s - min_s) * 42
    
    data = data.sort_values([COL_YEAR, COL_COUNTRY_CODE])
    years = sorted(data[COL_YEAR].unique())
    countries = data[COL_COUNTRY_CODE].unique()
    clusters = sorted(data['Cluster_1995'].unique())
    
    # Build country data dictionary
    country_data = {}
    for code in countries:
        cdf = data[data[COL_COUNTRY_CODE] == code].sort_values(COL_YEAR)
        
        # Get 1995 origin position
        origin_row = cdf[cdf[COL_YEAR] == 1995]
        if len(origin_row) == 0:
            continue  # Skip countries without 1995 data
            
        country_data[code] = {
            'years': cdf[COL_YEAR].values,
            'x': cdf['Log GDP per capita'].values,
            'y': cdf[COL_ECI].values,
            'x_origin': origin_row['Log GDP per capita'].values[0],
            'y_origin': origin_row[COL_ECI].values[0],
            'size': cdf['Bubble_Size_Scaled'].values,
            'name': cdf[COL_COUNTRY_NAME].iloc[0],
            'cluster': cdf['Cluster_1995'].iloc[0],
            'prod_pc': cdf['Production_Per_Capita'].values
        }
    
    countries = list(country_data.keys())
    
    # ============ BUILD FIGURE ============
    fig = go.Figure()
    
    first_year = years[0]
    
    for cluster in clusters:
        cluster_countries = [c for c in countries if country_data[c]['cluster'] == cluster]
        color = CLUSTER_COLORS[cluster]
        
        # Add ARROW traces
        for code in cluster_countries:
            cd = country_data[code]
            idx = np.where(cd['years'] == first_year)[0]
            
            if len(idx) > 0:
                i = idx[0]
                x_current, y_current = cd['x'][i], cd['y'][i]
            else:
                x_current, y_current = cd['x_origin'], cd['y_origin']
            
            fig.add_trace(go.Scatter(
                x=[cd['x_origin'], x_current],
                y=[cd['y_origin'], y_current],
                mode='lines',
                line=dict(color=color, width=arrow_width),
                opacity=arrow_opacity,
                legendgroup=f"cluster_{cluster}",
                showlegend=False,
                hoverinfo='skip'
            ))
        
        # Add BUBBLE traces
        for code in cluster_countries:
            cd = country_data[code]
            idx = np.where(cd['years'] == first_year)[0]
            
            if len(idx) > 0:
                i = idx[0]
                x_val, y_val = [cd['x'][i]], [cd['y'][i]]
                size_val, prod_val = cd['size'][i], cd['prod_pc'][i]
            else:
                x_val, y_val = [cd['x_origin']], [cd['y_origin']]
                size_val, prod_val = 15, 0
            
            fig.add_trace(go.Scatter(
                x=x_val, y=y_val,
                mode='markers+text',
                marker=dict(size=size_val, color=color, opacity=0.85,
                            line=dict(width=1.5, color='white')),
                text=[code],
                textposition='top center',
                textfont=dict(size=8, color='black'),
                name=CLUSTER_NAMES[cluster],
                legendgroup=f"cluster_{cluster}",
                showlegend=(code == cluster_countries[0]),
                customdata=[[cd['name'], prod_val, first_year]],
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Log GDP pc: %{x:.2f}<br>"
                    "ECI: %{y:.2f}<br>"
                    "Prod/capita: $%{customdata[1]:,.0f}<br>"
                    "Year: %{customdata[2]}<extra></extra>"
                )
            ))
        
        # Add ORIGIN MARKERS
        for code in cluster_countries:
            cd = country_data[code]
            fig.add_trace(go.Scatter(
                x=[cd['x_origin']],
                y=[cd['y_origin']],
                mode='markers',
                marker=dict(size=5, color=color, opacity=0.6, symbol='circle'),
                legendgroup=f"cluster_{cluster}",
                showlegend=False,
                hoverinfo='skip'
            ))
    
    # ============ BUILD FRAMES ============
    frames = []
    for year in years:
        frame_data = []
        
        for cluster in clusters:
            cluster_countries = [c for c in countries if country_data[c]['cluster'] == cluster]
            color = CLUSTER_COLORS[cluster]
            
            for code in cluster_countries:
                cd = country_data[code]
                idx = np.where(cd['years'] == year)[0]
                
                if len(idx) > 0:
                    i = idx[0]
                    x_current, y_current = cd['x'][i], cd['y'][i]
                else:
                    valid_mask = cd['years'] <= year
                    if valid_mask.any():
                        last_idx = np.where(valid_mask)[0][-1]
                        x_current, y_current = cd['x'][last_idx], cd['y'][last_idx]
                    else:
                        x_current, y_current = cd['x_origin'], cd['y_origin']
                
                frame_data.append(go.Scatter(
                    x=[cd['x_origin'], x_current],
                    y=[cd['y_origin'], y_current],
                    mode='lines',
                    line=dict(color=color, width=arrow_width),
                    opacity=arrow_opacity
                ))
            
            for code in cluster_countries:
                cd = country_data[code]
                idx = np.where(cd['years'] == year)[0]
                
                if len(idx) > 0:
                    i = idx[0]
                    x_val, y_val = [cd['x'][i]], [cd['y'][i]]
                    size_val, prod_val = cd['size'][i], cd['prod_pc'][i]
                else:
                    valid_mask = cd['years'] <= year
                    if valid_mask.any():
                        last_idx = np.where(valid_mask)[0][-1]
                        x_val, y_val = [cd['x'][last_idx]], [cd['y'][last_idx]]
                        size_val, prod_val = cd['size'][last_idx], cd['prod_pc'][last_idx]
                    else:
                        x_val, y_val = [cd['x_origin']], [cd['y_origin']]
                        size_val, prod_val = 15, 0
                
                frame_data.append(go.Scatter(
                    x=x_val, y=y_val,
                    mode='markers+text',
                    marker=dict(size=size_val, color=color, opacity=0.85,
                                line=dict(width=1.5, color='white')),
                    text=[code],
                    textposition='top center',
                    textfont=dict(size=8, color='black'),
                    customdata=[[cd['name'], prod_val, year]],
                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Log GDP pc: %{x:.2f}<br>"
                        "ECI: %{y:.2f}<br>"
                        "Prod/capita: $%{customdata[1]:,.0f}<br>"
                        "Year: %{customdata[2]}<extra></extra>"
                    )
                ))
            
            for code in cluster_countries:
                cd = country_data[code]
                frame_data.append(go.Scatter(
                    x=[cd['x_origin']],
                    y=[cd['y_origin']],
                    mode='markers',
                    marker=dict(size=5, color=color, opacity=0.6, symbol='circle')
                ))
        
        frames.append(go.Frame(data=frame_data, name=str(year)))
    
    fig.frames = frames
    
    # ============ LAYOUT ============
    for eci_val in [-1, 0, 1]:
        fig.add_hline(y=eci_val, line_dash="dot",
                      line_color="rgba(150,150,150,0.4)", line_width=1)
    
    eci_min, eci_max = data[COL_ECI].min(), data[COL_ECI].max()
    x_min, x_max = data['Log GDP per capita'].min(), data['Log GDP per capita'].max()
    
    fig.update_layout(
        title=dict(
            text='Evolution of Economic Complexity vs Income<br>'
                 '<sup> Bubble size = Production per Capita</sup>',
            x=0.5, xanchor='center'
        ),
        xaxis=dict(range=[x_min - 0.2, x_max + 0.2],
                   title='Log GDP per capita (PPP)',
                   gridcolor='rgba(200,200,200,0.3)', showgrid=True),
        yaxis=dict(range=[eci_min - 0.5, eci_max + 0.5],
                   title='Economic Complexity Index',
                   gridcolor='rgba(200,200,200,0.3)', showgrid=True),
        plot_bgcolor='white',
        legend=dict(title='Resource Profile (1995)', x=1.02, y=0.99),
        width=850, height=650,
        updatemenus=[
            dict(type='buttons', showactive=True, x=1.0, y=-0.02,
                 xanchor='left', yanchor='top',
                 buttons=[
                     dict(label='▶', method='animate',
                          args=[None, dict(frame=dict(duration=500, redraw=True),
                                           fromcurrent=True,
                                           transition=dict(duration=300))]),
                     dict(label='⏸', method='animate',
                          args=[[None], dict(frame=dict(duration=0), mode='immediate')])
                 ])
        ],
        sliders=[{
            'active': 0,
            'currentvalue': {'prefix': 'Year: ', 'font': {'size': 14}, 'xanchor': 'left', 'offset': 10},
            'len': 0.85, 'x': 0.05, 'y': -0.12,
            'pad': {'t': 30},
            'steps': [dict(args=[[str(y)], dict(frame=dict(duration=300, redraw=True),
                                                mode='immediate')],
                           method='animate', label=str(y)) for y in years]
        }]
    )
    
    return fig


# ============ RUN ============
fig = create_rosling_chart_arrows(master, arrow_opacity=0.5, arrow_width=2)
fig.show()
fig.write_html(r"C:\Users\ayaan\Downloads\rosling_arrows.html")